# GAT on the entity graph (IEEE-CIS fraud)
**Pipeline:** the graph has only *entity* nodes (card1, addr1, P_emaildomain, R_emaildomain, DeviceInfo, DeviceType; 14,239 nodes). A payment is not a node: each payment row stores the 6 entity node ids it touches. The GAT learns an embedding for every entity node, and a payment is classified from the concatenation of its 6 entity embeddings.

**GAT stays graph-only: no tabular features.** Combining it with the tabular model happens in the separate fusion step, which uses the score files saved here.

**Runs (6):** real graph with seeds 42, 1, 2; shuffled graphs with seed 42 on the seed-42 file, seed 1 on the seed-1 file, seed 2 on the seed-2 file. The shuffled graphs are the control: same payments, entity links rebuilt after permuting each entity column.

**Training:** Adam, exactly 12 epochs, no early stopping, best epoch chosen by validation ROC-AUC. Each run saves val/test score files (raw logits, before the sigmoid) and a history JSON.

## 0. Setup

In [ ]:
!pip install -q torch_geometric

In [ ]:
import os, json, random, shutil, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_recall_curve, precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = '/content/drive/MyDrive/fraud_project/data/graph_files'
OUT_DIR  = '/content/drive/MyDrive/fraud_project/results'
os.makedirs(OUT_DIR, exist_ok=True)

GRAPH_FILES = {
    'real':   'graph_edges.npy',
    'shuf1':  'graph_shuf_seed1_edges.npy',
    'shuf2':  'graph_shuf_seed2_edges.npy',
    'shuf42': 'graph_shuf_seed42_edges.npy',
}
# (run name, seed, graph key)
RUNS = [
    ('gat_seed42',      42, 'real'),
    ('gat_seed1',       1,  'real'),
    ('gat_seed2',       2,  'real'),
    ('gat_shuf_seed42', 42, 'shuf42'),
    ('gat_shuf_seed1',  1,  'shuf1'),
    ('gat_shuf_seed2',  2,  'shuf2'),
]
RESULTS = {}   # filled by each run's Results cell

In [ ]:
# training settings (written into every history JSON)
LR, EPOCHS, BATCH_SIZE, UNK_PROB = 1e-3, 12, 4096, 0.10
EMB_DIM, HIDDEN, HEADS, DROPOUT, ATTN_DROPOUT = 64, 64, 4, 0.3, 0.2

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 1. Data

In [ ]:
info = json.load(open(f'{DATA_DIR}/graph_info.json'))
ENT_COLS = info['entity_columns']
NUM_NODES = info['num_nodes']
node_type = torch.from_numpy(np.load(f'{DATA_DIR}/node_type.npy')).long()
N_TYPES = int(node_type.max()) + 1
assert node_type.shape[0] == NUM_NODES

# one "unknown" node per entity column, in ENT_COLS order
UNK = torch.tensor([info['unknown_node'][c] for c in ENT_COLS], dtype=torch.long, device=device)
print(ENT_COLS, NUM_NODES, N_TYPES)
print('unknown nodes:', dict(zip(ENT_COLS, UNK.tolist())))

In [ ]:
train_df = pd.read_parquet(f'{DATA_DIR}/payment_nodes_train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/payment_nodes_val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/payment_nodes_test.parquet')

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(name, d.shape, 'fraud rate = %.4f' % d['isFraud'].mean())
train_df.head()

In [ ]:
def to_tensors(d):
    ent = torch.from_numpy(d[ENT_COLS].values.astype(np.int64))
    y   = torch.from_numpy(d['isFraud'].values.astype(np.float32))
    return ent.to(device), y.to(device)

ent_train, y_train = to_tensors(train_df)
ent_val,   y_val   = to_tensors(val_df)
ent_test,  y_test  = to_tensors(test_df)

for e in (ent_train, ent_val, ent_test):
    assert int(e.max()) < NUM_NODES and int(e.min()) >= 0

In [ ]:
def load_edges(path):
    e = torch.from_numpy(np.load(path)).long()   # [2, E], both directions already present
    assert e.shape[0] == 2
    return e.to(device)

# loss weight = sqrt(non-fraud / fraud)  (about 5.24), instead of the full ratio
fr = y_train.mean().item()
pos_weight = torch.tensor([((1 - fr) / fr) ** 0.5], device=device)
print('full ratio = %.1f | pos_weight (sqrt) = %.2f' % ((1 - fr) / fr, pos_weight.item()))

## 2. Model, Train (function), Validate (function)

In [ ]:
class GAT(nn.Module):
    """Learnable node embedding + type embedding -> 2 GAT layers -> payment head on the 6 entity embeddings."""
    def __init__(self, num_nodes, node_type, n_types, n_entities=6,
                 emb_dim=64, hidden=64, heads=4, dropout=0.3, attn_dropout=0.2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, emb_dim)
        self.type_emb = nn.Embedding(n_types, emb_dim)
        self.register_buffer('node_type', node_type)
        self.conv1 = GATConv(emb_dim, hidden, heads=heads, dropout=attn_dropout)       # -> hidden*heads
        self.conv2 = GATConv(hidden * heads, hidden, heads=1, dropout=attn_dropout)    # -> hidden
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden * n_entities, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, 1))

    def encode(self, edge_index):
        x = self.node_emb.weight + self.type_emb(self.node_type)
        x = self.drop(x)
        x = F.elu(self.conv1(x, edge_index))
        x = self.drop(x)
        return self.conv2(x, edge_index)                 # [num_nodes, hidden]

    def classify(self, h, ent):                          # ent: [B, 6] node ids
        return self.head(h[ent].flatten(1)).squeeze(-1)  # logits [B]

In [ ]:
def train_epoch(model, edge_index, ent, y, optimizer, criterion,
                batch_size=BATCH_SIZE, unk=UNK, unk_prob=UNK_PROB):
    model.train()
    perm = torch.randperm(ent.shape[0], device=ent.device)
    total = 0.0
    for i in range(0, len(perm), batch_size):
        idx = perm[i:i + batch_size]
        e = ent[idx]
        if unk_prob > 0:   # training only: each entity -> its column's unknown node with prob unk_prob
            drop = torch.rand(e.shape, device=e.device) < unk_prob
            e = torch.where(drop, unk.expand_as(e), e)
        h = model.encode(edge_index)                     # graph is small, re-encode each step
        loss = criterion(model.classify(h, e), y[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(idx)
    return total / len(perm)

In [ ]:
@torch.no_grad()
def validate(model, edge_index, ent, y, criterion, batch_size=65536):
    model.eval()                                         # no unknown-node replacement here
    h = model.encode(edge_index)
    logits = torch.cat([model.classify(h, ent[i:i + batch_size])
                        for i in range(0, ent.shape[0], batch_size)])
    loss = criterion(logits, y).item()
    yy = y.cpu().numpy()
    lg = logits.cpu().numpy().astype(np.float32)         # raw scores, before the sigmoid
    return {'loss': loss,
            'auprc': average_precision_score(yy, lg),    # rank-based, so same as on probabilities
            'auroc': roc_auc_score(yy, lg),
            'logits': lg,
            'probs': torch.sigmoid(logits).cpu().numpy()}

In [ ]:
def best_threshold(y_true, probs):
    p, r, t = precision_recall_curve(y_true, probs)
    f1 = 2 * p * r / (p + r + 1e-12)
    return float(t[np.argmax(f1[:-1])])

def test_report(y_true, probs, thr):
    pred = (probs >= thr).astype(int)
    return {'AUPRC': average_precision_score(y_true, probs),
            'AUROC': roc_auc_score(y_true, probs),
            'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred),
            'F1': f1_score(y_true, pred),
            'Threshold': thr}

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
def save_scores(run, split, df, logits):
    out = pd.DataFrame({'TransactionID': df['TransactionID'].values,
                        'isFraud': df['isFraud'].values,
                        'score': logits.astype(np.float32)})
    path = f'{OUT_DIR}/{run}_{split}.parquet'
    out.to_parquet(path, index=False)
    print('saved', path, out.shape)

def save_history(run, seed, graph_key, hist, best_epoch):
    rec = {'run': run, 'seed': seed, 'graph_file': GRAPH_FILES[graph_key],
           'settings': {'optimizer': 'Adam', 'lr': LR, 'epochs': EPOCHS, 'early_stopping': False,
                        'batch_size': BATCH_SIZE, 'unknown_node_prob': UNK_PROB,
                        'loss_pos_weight': round(pos_weight.item(), 4), 'loss_pos_weight_rule': 'sqrt(non-fraud/fraud)',
                        'emb_dim': EMB_DIM, 'hidden': HIDDEN, 'heads': HEADS,
                        'dropout': DROPOUT, 'attn_dropout': ATTN_DROPOUT,
                        'model_selection': 'best validation ROC-AUC', 'score': 'logit (before sigmoid)'},
           'best_epoch': best_epoch,
           'epochs': hist}
    path = f'{OUT_DIR}/{run}_history.json'
    json.dump(rec, open(path, 'w'), indent=2)
    print('saved', path)

# 3. Real graph (3 seeds)

## Run: gat_seed42  (seed 42, graph: real)

### gat_seed42 - Setup

In [ ]:
set_seed(42)                      # seed set right before the model is created
edge_index_real42 = load_edges(f"{DATA_DIR}/{GRAPH_FILES['real']}")
print('graph:', GRAPH_FILES['real'], tuple(edge_index_real42.shape))
model_real42 = GAT(NUM_NODES, node_type.to(device), N_TYPES, emb_dim=EMB_DIM, hidden=HIDDEN,
                  heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT).to(device)
opt_real42 = torch.optim.Adam(model_real42.parameters(), lr=LR)
print(sum(p.numel() for p in model_real42.parameters()), 'parameters')

### gat_seed42 - Train

In [ ]:
hist_real42 = []
best_real42 = {'auroc': -1, 'epoch': 0, 'state': None}
for epoch in range(1, EPOCHS + 1):                  # exactly EPOCHS epochs, no early stopping
    tr_loss = train_epoch(model_real42, edge_index_real42, ent_train, y_train, opt_real42, criterion)
    va = validate(model_real42, edge_index_real42, ent_val, y_val, criterion)
    hist_real42.append({'epoch': epoch, 'train_loss': tr_loss,
                       'val_roc_auc': va['auroc'], 'val_pr_auc': va['auprc']})
    print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} | val ROC-AUC {va['auroc']:.4f} | val PR-AUC {va['auprc']:.4f}")
    if va['auroc'] > best_real42['auroc']:
        best_real42 = {'auroc': va['auroc'], 'epoch': epoch,
                      'state': {k: v.detach().clone() for k, v in model_real42.state_dict().items()}}

### gat_seed42 - Best Epoch

In [ ]:
h_real42 = pd.DataFrame(hist_real42)
print('best epoch:', best_real42['epoch'], '| val ROC-AUC: %.4f' % best_real42['auroc'])
model_real42.load_state_dict(best_real42['state'])    # everything below uses this epoch

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_real42.epoch, h_real42.train_loss); ax[0].set_title('Train loss')
ax[1].plot(h_real42.epoch, h_real42.val_roc_auc, label='val ROC-AUC')
ax[1].plot(h_real42.epoch, h_real42.val_pr_auc, label='val PR-AUC')
ax[1].axvline(best_real42['epoch'], ls='--', c='gray'); ax[1].set_title('Validation'); ax[1].legend()
plt.tight_layout(); plt.show()

### gat_seed42 - Results

In [ ]:
va_real42 = validate(model_real42, edge_index_real42, ent_val, y_val, criterion)
te_real42 = validate(model_real42, edge_index_real42, ent_test, y_test, criterion)
thr = best_threshold(y_val.cpu().numpy(), va_real42['probs'])     # threshold picked on validation only
rep = test_report(y_test.cpu().numpy(), te_real42['probs'], thr)
RESULTS['gat_seed42'] = {'graph': 'real', 'seed': 42, 'best_epoch': best_real42['epoch'], **rep}
pd.Series(RESULTS['gat_seed42'])

### gat_seed42 - Save scores and history

In [ ]:
save_scores('gat_seed42', 'val',  val_df,  va_real42['logits'])
save_scores('gat_seed42', 'test', test_df, te_real42['logits'])
save_history('gat_seed42', 42, 'real', hist_real42, best_real42['epoch'])

## Run: gat_seed1  (seed 1, graph: real)

### gat_seed1 - Setup

In [ ]:
set_seed(1)                      # seed set right before the model is created
edge_index_real1 = load_edges(f"{DATA_DIR}/{GRAPH_FILES['real']}")
print('graph:', GRAPH_FILES['real'], tuple(edge_index_real1.shape))
model_real1 = GAT(NUM_NODES, node_type.to(device), N_TYPES, emb_dim=EMB_DIM, hidden=HIDDEN,
                  heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT).to(device)
opt_real1 = torch.optim.Adam(model_real1.parameters(), lr=LR)
print(sum(p.numel() for p in model_real1.parameters()), 'parameters')

### gat_seed1 - Train

In [ ]:
hist_real1 = []
best_real1 = {'auroc': -1, 'epoch': 0, 'state': None}
for epoch in range(1, EPOCHS + 1):                  # exactly EPOCHS epochs, no early stopping
    tr_loss = train_epoch(model_real1, edge_index_real1, ent_train, y_train, opt_real1, criterion)
    va = validate(model_real1, edge_index_real1, ent_val, y_val, criterion)
    hist_real1.append({'epoch': epoch, 'train_loss': tr_loss,
                       'val_roc_auc': va['auroc'], 'val_pr_auc': va['auprc']})
    print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} | val ROC-AUC {va['auroc']:.4f} | val PR-AUC {va['auprc']:.4f}")
    if va['auroc'] > best_real1['auroc']:
        best_real1 = {'auroc': va['auroc'], 'epoch': epoch,
                      'state': {k: v.detach().clone() for k, v in model_real1.state_dict().items()}}

### gat_seed1 - Best Epoch

In [ ]:
h_real1 = pd.DataFrame(hist_real1)
print('best epoch:', best_real1['epoch'], '| val ROC-AUC: %.4f' % best_real1['auroc'])
model_real1.load_state_dict(best_real1['state'])    # everything below uses this epoch

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_real1.epoch, h_real1.train_loss); ax[0].set_title('Train loss')
ax[1].plot(h_real1.epoch, h_real1.val_roc_auc, label='val ROC-AUC')
ax[1].plot(h_real1.epoch, h_real1.val_pr_auc, label='val PR-AUC')
ax[1].axvline(best_real1['epoch'], ls='--', c='gray'); ax[1].set_title('Validation'); ax[1].legend()
plt.tight_layout(); plt.show()

### gat_seed1 - Results

In [ ]:
va_real1 = validate(model_real1, edge_index_real1, ent_val, y_val, criterion)
te_real1 = validate(model_real1, edge_index_real1, ent_test, y_test, criterion)
thr = best_threshold(y_val.cpu().numpy(), va_real1['probs'])     # threshold picked on validation only
rep = test_report(y_test.cpu().numpy(), te_real1['probs'], thr)
RESULTS['gat_seed1'] = {'graph': 'real', 'seed': 1, 'best_epoch': best_real1['epoch'], **rep}
pd.Series(RESULTS['gat_seed1'])

### gat_seed1 - Save scores and history

In [ ]:
save_scores('gat_seed1', 'val',  val_df,  va_real1['logits'])
save_scores('gat_seed1', 'test', test_df, te_real1['logits'])
save_history('gat_seed1', 1, 'real', hist_real1, best_real1['epoch'])

## Run: gat_seed2  (seed 2, graph: real)

### gat_seed2 - Setup

In [ ]:
set_seed(2)                      # seed set right before the model is created
edge_index_real2 = load_edges(f"{DATA_DIR}/{GRAPH_FILES['real']}")
print('graph:', GRAPH_FILES['real'], tuple(edge_index_real2.shape))
model_real2 = GAT(NUM_NODES, node_type.to(device), N_TYPES, emb_dim=EMB_DIM, hidden=HIDDEN,
                  heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT).to(device)
opt_real2 = torch.optim.Adam(model_real2.parameters(), lr=LR)
print(sum(p.numel() for p in model_real2.parameters()), 'parameters')

### gat_seed2 - Train

In [ ]:
hist_real2 = []
best_real2 = {'auroc': -1, 'epoch': 0, 'state': None}
for epoch in range(1, EPOCHS + 1):                  # exactly EPOCHS epochs, no early stopping
    tr_loss = train_epoch(model_real2, edge_index_real2, ent_train, y_train, opt_real2, criterion)
    va = validate(model_real2, edge_index_real2, ent_val, y_val, criterion)
    hist_real2.append({'epoch': epoch, 'train_loss': tr_loss,
                       'val_roc_auc': va['auroc'], 'val_pr_auc': va['auprc']})
    print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} | val ROC-AUC {va['auroc']:.4f} | val PR-AUC {va['auprc']:.4f}")
    if va['auroc'] > best_real2['auroc']:
        best_real2 = {'auroc': va['auroc'], 'epoch': epoch,
                      'state': {k: v.detach().clone() for k, v in model_real2.state_dict().items()}}

### gat_seed2 - Best Epoch

In [ ]:
h_real2 = pd.DataFrame(hist_real2)
print('best epoch:', best_real2['epoch'], '| val ROC-AUC: %.4f' % best_real2['auroc'])
model_real2.load_state_dict(best_real2['state'])    # everything below uses this epoch

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_real2.epoch, h_real2.train_loss); ax[0].set_title('Train loss')
ax[1].plot(h_real2.epoch, h_real2.val_roc_auc, label='val ROC-AUC')
ax[1].plot(h_real2.epoch, h_real2.val_pr_auc, label='val PR-AUC')
ax[1].axvline(best_real2['epoch'], ls='--', c='gray'); ax[1].set_title('Validation'); ax[1].legend()
plt.tight_layout(); plt.show()

### gat_seed2 - Results

In [ ]:
va_real2 = validate(model_real2, edge_index_real2, ent_val, y_val, criterion)
te_real2 = validate(model_real2, edge_index_real2, ent_test, y_test, criterion)
thr = best_threshold(y_val.cpu().numpy(), va_real2['probs'])     # threshold picked on validation only
rep = test_report(y_test.cpu().numpy(), te_real2['probs'], thr)
RESULTS['gat_seed2'] = {'graph': 'real', 'seed': 2, 'best_epoch': best_real2['epoch'], **rep}
pd.Series(RESULTS['gat_seed2'])

### gat_seed2 - Save scores and history

In [ ]:
save_scores('gat_seed2', 'val',  val_df,  va_real2['logits'])
save_scores('gat_seed2', 'test', test_df, te_real2['logits'])
save_history('gat_seed2', 2, 'real', hist_real2, best_real2['epoch'])

# 4. Controls: shuffled graphs (seed matches the graph file's seed)

## Run: gat_shuf_seed42  (seed 42, graph: shuf42)

### gat_shuf_seed42 - Setup

In [ ]:
set_seed(42)                      # seed set right before the model is created
edge_index_shuf42 = load_edges(f"{DATA_DIR}/{GRAPH_FILES['shuf42']}")
print('graph:', GRAPH_FILES['shuf42'], tuple(edge_index_shuf42.shape))
model_shuf42 = GAT(NUM_NODES, node_type.to(device), N_TYPES, emb_dim=EMB_DIM, hidden=HIDDEN,
                  heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT).to(device)
opt_shuf42 = torch.optim.Adam(model_shuf42.parameters(), lr=LR)
print(sum(p.numel() for p in model_shuf42.parameters()), 'parameters')

### gat_shuf_seed42 - Train

In [ ]:
hist_shuf42 = []
best_shuf42 = {'auroc': -1, 'epoch': 0, 'state': None}
for epoch in range(1, EPOCHS + 1):                  # exactly EPOCHS epochs, no early stopping
    tr_loss = train_epoch(model_shuf42, edge_index_shuf42, ent_train, y_train, opt_shuf42, criterion)
    va = validate(model_shuf42, edge_index_shuf42, ent_val, y_val, criterion)
    hist_shuf42.append({'epoch': epoch, 'train_loss': tr_loss,
                       'val_roc_auc': va['auroc'], 'val_pr_auc': va['auprc']})
    print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} | val ROC-AUC {va['auroc']:.4f} | val PR-AUC {va['auprc']:.4f}")
    if va['auroc'] > best_shuf42['auroc']:
        best_shuf42 = {'auroc': va['auroc'], 'epoch': epoch,
                      'state': {k: v.detach().clone() for k, v in model_shuf42.state_dict().items()}}

### gat_shuf_seed42 - Best Epoch

In [ ]:
h_shuf42 = pd.DataFrame(hist_shuf42)
print('best epoch:', best_shuf42['epoch'], '| val ROC-AUC: %.4f' % best_shuf42['auroc'])
model_shuf42.load_state_dict(best_shuf42['state'])    # everything below uses this epoch

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_shuf42.epoch, h_shuf42.train_loss); ax[0].set_title('Train loss')
ax[1].plot(h_shuf42.epoch, h_shuf42.val_roc_auc, label='val ROC-AUC')
ax[1].plot(h_shuf42.epoch, h_shuf42.val_pr_auc, label='val PR-AUC')
ax[1].axvline(best_shuf42['epoch'], ls='--', c='gray'); ax[1].set_title('Validation'); ax[1].legend()
plt.tight_layout(); plt.show()

### gat_shuf_seed42 - Results

In [ ]:
va_shuf42 = validate(model_shuf42, edge_index_shuf42, ent_val, y_val, criterion)
te_shuf42 = validate(model_shuf42, edge_index_shuf42, ent_test, y_test, criterion)
thr = best_threshold(y_val.cpu().numpy(), va_shuf42['probs'])     # threshold picked on validation only
rep = test_report(y_test.cpu().numpy(), te_shuf42['probs'], thr)
RESULTS['gat_shuf_seed42'] = {'graph': 'shuffled', 'seed': 42, 'best_epoch': best_shuf42['epoch'], **rep}
pd.Series(RESULTS['gat_shuf_seed42'])

### gat_shuf_seed42 - Save scores and history

In [ ]:
save_scores('gat_shuf_seed42', 'val',  val_df,  va_shuf42['logits'])
save_scores('gat_shuf_seed42', 'test', test_df, te_shuf42['logits'])
save_history('gat_shuf_seed42', 42, 'shuf42', hist_shuf42, best_shuf42['epoch'])

## Run: gat_shuf_seed1  (seed 1, graph: shuf1)

### gat_shuf_seed1 - Setup

In [ ]:
set_seed(1)                      # seed set right before the model is created
edge_index_shuf1 = load_edges(f"{DATA_DIR}/{GRAPH_FILES['shuf1']}")
print('graph:', GRAPH_FILES['shuf1'], tuple(edge_index_shuf1.shape))
model_shuf1 = GAT(NUM_NODES, node_type.to(device), N_TYPES, emb_dim=EMB_DIM, hidden=HIDDEN,
                  heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT).to(device)
opt_shuf1 = torch.optim.Adam(model_shuf1.parameters(), lr=LR)
print(sum(p.numel() for p in model_shuf1.parameters()), 'parameters')

### gat_shuf_seed1 - Train

In [ ]:
hist_shuf1 = []
best_shuf1 = {'auroc': -1, 'epoch': 0, 'state': None}
for epoch in range(1, EPOCHS + 1):                  # exactly EPOCHS epochs, no early stopping
    tr_loss = train_epoch(model_shuf1, edge_index_shuf1, ent_train, y_train, opt_shuf1, criterion)
    va = validate(model_shuf1, edge_index_shuf1, ent_val, y_val, criterion)
    hist_shuf1.append({'epoch': epoch, 'train_loss': tr_loss,
                       'val_roc_auc': va['auroc'], 'val_pr_auc': va['auprc']})
    print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} | val ROC-AUC {va['auroc']:.4f} | val PR-AUC {va['auprc']:.4f}")
    if va['auroc'] > best_shuf1['auroc']:
        best_shuf1 = {'auroc': va['auroc'], 'epoch': epoch,
                      'state': {k: v.detach().clone() for k, v in model_shuf1.state_dict().items()}}

### gat_shuf_seed1 - Best Epoch

In [ ]:
h_shuf1 = pd.DataFrame(hist_shuf1)
print('best epoch:', best_shuf1['epoch'], '| val ROC-AUC: %.4f' % best_shuf1['auroc'])
model_shuf1.load_state_dict(best_shuf1['state'])    # everything below uses this epoch

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_shuf1.epoch, h_shuf1.train_loss); ax[0].set_title('Train loss')
ax[1].plot(h_shuf1.epoch, h_shuf1.val_roc_auc, label='val ROC-AUC')
ax[1].plot(h_shuf1.epoch, h_shuf1.val_pr_auc, label='val PR-AUC')
ax[1].axvline(best_shuf1['epoch'], ls='--', c='gray'); ax[1].set_title('Validation'); ax[1].legend()
plt.tight_layout(); plt.show()

### gat_shuf_seed1 - Results

In [ ]:
va_shuf1 = validate(model_shuf1, edge_index_shuf1, ent_val, y_val, criterion)
te_shuf1 = validate(model_shuf1, edge_index_shuf1, ent_test, y_test, criterion)
thr = best_threshold(y_val.cpu().numpy(), va_shuf1['probs'])     # threshold picked on validation only
rep = test_report(y_test.cpu().numpy(), te_shuf1['probs'], thr)
RESULTS['gat_shuf_seed1'] = {'graph': 'shuffled', 'seed': 1, 'best_epoch': best_shuf1['epoch'], **rep}
pd.Series(RESULTS['gat_shuf_seed1'])

### gat_shuf_seed1 - Save scores and history

In [ ]:
save_scores('gat_shuf_seed1', 'val',  val_df,  va_shuf1['logits'])
save_scores('gat_shuf_seed1', 'test', test_df, te_shuf1['logits'])
save_history('gat_shuf_seed1', 1, 'shuf1', hist_shuf1, best_shuf1['epoch'])

## Run: gat_shuf_seed2  (seed 2, graph: shuf2)

### gat_shuf_seed2 - Setup

In [ ]:
set_seed(2)                      # seed set right before the model is created
edge_index_shuf2 = load_edges(f"{DATA_DIR}/{GRAPH_FILES['shuf2']}")
print('graph:', GRAPH_FILES['shuf2'], tuple(edge_index_shuf2.shape))
model_shuf2 = GAT(NUM_NODES, node_type.to(device), N_TYPES, emb_dim=EMB_DIM, hidden=HIDDEN,
                  heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT).to(device)
opt_shuf2 = torch.optim.Adam(model_shuf2.parameters(), lr=LR)
print(sum(p.numel() for p in model_shuf2.parameters()), 'parameters')

### gat_shuf_seed2 - Train

In [ ]:
hist_shuf2 = []
best_shuf2 = {'auroc': -1, 'epoch': 0, 'state': None}
for epoch in range(1, EPOCHS + 1):                  # exactly EPOCHS epochs, no early stopping
    tr_loss = train_epoch(model_shuf2, edge_index_shuf2, ent_train, y_train, opt_shuf2, criterion)
    va = validate(model_shuf2, edge_index_shuf2, ent_val, y_val, criterion)
    hist_shuf2.append({'epoch': epoch, 'train_loss': tr_loss,
                       'val_roc_auc': va['auroc'], 'val_pr_auc': va['auprc']})
    print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} | val ROC-AUC {va['auroc']:.4f} | val PR-AUC {va['auprc']:.4f}")
    if va['auroc'] > best_shuf2['auroc']:
        best_shuf2 = {'auroc': va['auroc'], 'epoch': epoch,
                      'state': {k: v.detach().clone() for k, v in model_shuf2.state_dict().items()}}

### gat_shuf_seed2 - Best Epoch

In [ ]:
h_shuf2 = pd.DataFrame(hist_shuf2)
print('best epoch:', best_shuf2['epoch'], '| val ROC-AUC: %.4f' % best_shuf2['auroc'])
model_shuf2.load_state_dict(best_shuf2['state'])    # everything below uses this epoch

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_shuf2.epoch, h_shuf2.train_loss); ax[0].set_title('Train loss')
ax[1].plot(h_shuf2.epoch, h_shuf2.val_roc_auc, label='val ROC-AUC')
ax[1].plot(h_shuf2.epoch, h_shuf2.val_pr_auc, label='val PR-AUC')
ax[1].axvline(best_shuf2['epoch'], ls='--', c='gray'); ax[1].set_title('Validation'); ax[1].legend()
plt.tight_layout(); plt.show()

### gat_shuf_seed2 - Results

In [ ]:
va_shuf2 = validate(model_shuf2, edge_index_shuf2, ent_val, y_val, criterion)
te_shuf2 = validate(model_shuf2, edge_index_shuf2, ent_test, y_test, criterion)
thr = best_threshold(y_val.cpu().numpy(), va_shuf2['probs'])     # threshold picked on validation only
rep = test_report(y_test.cpu().numpy(), te_shuf2['probs'], thr)
RESULTS['gat_shuf_seed2'] = {'graph': 'shuffled', 'seed': 2, 'best_epoch': best_shuf2['epoch'], **rep}
pd.Series(RESULTS['gat_shuf_seed2'])

### gat_shuf_seed2 - Save scores and history

In [ ]:
save_scores('gat_shuf_seed2', 'val',  val_df,  va_shuf2['logits'])
save_scores('gat_shuf_seed2', 'test', test_df, te_shuf2['logits'])
save_history('gat_shuf_seed2', 2, 'shuf2', hist_shuf2, best_shuf2['epoch'])

# 5. Summary table and download

In [ ]:
res = pd.DataFrame.from_dict(RESULTS, orient='index').round(4)
res.index.name = 'run'
res.to_csv(f'{OUT_DIR}/gat_results.csv')
res

In [ ]:
# mean and std over the 3 seeds, per graph type (numbers only)
res.groupby('graph')[['AUPRC', 'AUROC', 'F1']].agg(['mean', 'std']).round(4)

In [ ]:
# check all 18 files exist, then zip them for download
expected = [f'{r}_{s}.parquet' for r, _, _ in RUNS for s in ('val', 'test')] + \
           [f'{r}_history.json' for r, _, _ in RUNS]
missing = [f for f in expected if not os.path.exists(f'{OUT_DIR}/{f}')]
assert not missing, f'missing: {missing}'
print(len(expected), 'files present')

os.makedirs('/content/gat_outputs', exist_ok=True)
for f in expected:
    shutil.copy(f'{OUT_DIR}/{f}', f'/content/gat_outputs/{f}')
zip_path = shutil.make_archive('/content/gat_outputs', 'zip', '/content/gat_outputs')
from google.colab import files
files.download(zip_path)